# 08 — Advanced SQL

## Overview

This notebook applies advanced SQL techniques to the movie database and Analytics Marts.

The objective is to develop analytical SQL skills required to transform, analyze, rank, compare, and validate data efficiently at the database level.

The notebook focuses on:

- Common Table Expressions (CTEs)
- Subqueries
- CASE expressions
- Conditional aggregation
- Window functions
- Ranking
- Running totals
- Period-over-period analysis
- Top-N analysis
- EXISTS and NOT EXISTS
- Set operations
- Advanced JOIN patterns
- Date-based analysis
- Financial analytics
- Query optimization
- Query validation

## Analytical Principle

Complex analytical problems should be solved by understanding:

1. Dataset grain
2. Business logic
3. Required transformation
4. Appropriate SQL operation
5. Expected result
6. Query performance

The objective is not simply to write SQL that returns a result.

The objective is to write SQL that is:

- Correct
- Explainable
- Efficient
- Maintainable
- Scalable

# Environment Setup

In [14]:
# Importing modules
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

print("Modules Imported Successfully")
print("SQLAlchemy Engine Created Successfully")


Modules Imported Successfully
SQLAlchemy Engine Created Successfully


In [15]:
# creating a connection to the SQLite database using SQLAlchemy
db_path = Path(r"C:\\Users\\user\\Documents\\Movie DB SQL Analytics\\Database\\TMDB-a-4006.db")

if not db_path.is_file():
    raise FileNotFoundError(f"Database file not found: {db_path}")

engine = create_engine(f"sqlite:///{db_path.as_posix()}")

print("SQLAlchemy Engine Created Successfully")

SQLAlchemy Engine Created Successfully


In [16]:
# Testing the database connection
with engine.connect() as connection:
    result = pd.read_sql(
        text("SELECT 1 AS Connection_Test"),
        connection
    )

print(result)
print("Database connection test successful!")


   Connection_Test
0                1
Database connection test successful!


In [17]:
# Retrieving the list of tables in the database
query = text("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""")

with engine.connect() as connection:
    tables = pd.read_sql(query, connection)

print(tables)
print("Tables retrieved successfully!")

                    name
0                 actors
1                  casts
2               genremap
3                 genres
4             keywordmap
5               keywords
6            languagemap
7              languages
8                 movies
9                 oscars
10   productioncompanies
11  productioncompanymap
12   productioncountries
13  productioncountrymap
14           sysdiagrams
Tables retrieved successfully!


In [18]:
# Querying the top 20 movies by profit
query = """
WITH movie_financials AS (
    SELECT
        movie_id,
        title,
        budget,
        revenue,
        revenue - budget AS profit
    FROM movies
)

SELECT *
FROM movie_financials
ORDER BY profit DESC
LIMIT 20;
"""

pd.read_sql(query, engine)

,movie_id,title,budget,revenue,profit
0,19995,Avatar,237000000,2.787965e+09,2.550965e+09
1,597,Titanic,200000000,1.845034e+09,1.645034e+09
2,135397,Jurassic World,150000000,1.513529e+09,1.363529e+09
3,168259,Furious 7,190000000,1.506249e+09,1.316249e+09
4,24428,The Avengers,220000000,1.519558e+09,1.299558e+09
5,99861,Avengers: Age of Ultron,280000000,1.405404e+09,1.125404e+09
6,109445,Frozen,150000000,1.274219e+09,1.124219e+09
7,211672,Minions,74000000,1.156731e+09,1.082731e+09
8,122,The Lord of the Rings: The Return of the King,94000000,1.118889e+09,1.024889e+09
9,68721,Iron Man 3,200000000,1.215440e+09,1.015440e+09


In [19]:
# Querying the top 20 movies by ROI (Return on Investment)
query = """
WITH movie_financials AS (

    SELECT
        movie_id,
        title,
        budget,
        revenue,
        revenue - budget AS profit
    FROM movies

),

movie_metrics AS (

    SELECT
        *,
        CASE
            WHEN budget > 0
            THEN (profit * 1.0 / budget) * 100
            ELSE NULL
        END AS roi_percentage
    FROM movie_financials

)

SELECT *
FROM movie_metrics
ORDER BY roi_percentage DESC
LIMIT 20;
"""

pd.read_sql(query, engine)

,movie_id,title,budget,revenue,profit,roi_percentage
0,3082,Modern Times,1,8500000.0,8499999.0,8.499999e+08
1,78383,Nurse 3-D,10,10000000.0,9999990.0,9.999990e+07
2,23827,Paranormal Activity,15000,193355800.0,193340800.0,1.288939e+06
3,1435,Tarnation,218,1162014.0,1161796.0,5.329339e+05
4,2667,The Blair Witch Project,60000,248000000.0,247940000.0,4.132333e+05
5,985,Eraserhead,10000,7000000.0,6990000.0,6.990000e+04
6,692,Pink Flamingos,12000,6000000.0,5988000.0,4.990000e+04
7,9372,Super Size Me,65000,28575078.0,28510078.0,4.386166e+04
8,299245,The Gallows,100000,42664410.0,42564410.0,4.256441e+04
9,83,Open Water,130000,54667954.0,54537954.0,4.195227e+04


In [20]:
# Querying the profitability status of movies
query = """
SELECT
    movie_id,
    title,
    budget,
    revenue,

    CASE
        WHEN revenue > budget THEN 'Profitable'
        WHEN revenue = budget THEN 'Break-even'
        WHEN revenue < budget THEN 'Loss'
        ELSE 'Unknown'
    END AS profitability_status

FROM movies;
"""

movie_profitability = pd.read_sql(query, engine)

movie_profitability.head()

,movie_id,title,budget,revenue,profitability_status
0,5,Four Rooms,4000000,4300000.0,Profitable
1,11,Star Wars,11000000,775398007.0,Profitable
2,12,Finding Nemo,94000000,940335536.0,Profitable
3,13,Forrest Gump,55000000,677945399.0,Profitable
4,14,American Beauty,15000000,356296601.0,Profitable


In [21]:
# Querying the budget category of movies
query = """
SELECT
    movie_id,
    title,
    budget,
    revenue,

    CASE
        WHEN budget = 0 THEN 'No Budget'
        WHEN budget < 1000000 THEN 'Low Budget'
        WHEN budget < 10000000 THEN 'Medium Budget'
        WHEN budget < 50000000 THEN 'High Budget'
        ELSE 'Very High Budget'
    END AS budget_category

FROM movies;
"""

pd.read_sql(query, engine)

,movie_id,title,budget,revenue,budget_category
0,5,Four Rooms,4000000,4300000.0,Medium Budget
1,11,Star Wars,11000000,775398007.0,High Budget
2,12,Finding Nemo,94000000,940335536.0,Very High Budget
3,13,Forrest Gump,55000000,677945399.0,Very High Budget
4,14,American Beauty,15000000,356296601.0,High Budget
...,...,...,...,...,...
4798,426067,Midnight Cabaret,0,0.0,No Budget
4799,426469,Growing Up Smith,0,0.0,No Budget
4800,433715,8 Days,0,0.0,No Budget
4801,447027,Running Forever,0,0.0,No Budget


In [22]:
# Querying the overall profitability statistics of movies
query = """
SELECT
    COUNT(*) AS Total_Movies,

    SUM(
        CASE
            WHEN revenue > budget THEN 1
            ELSE 0
        END
    ) AS Profitable_Movies,

    SUM(
        CASE
            WHEN revenue < budget THEN 1
            ELSE 0
        END
    ) AS Loss_Making_Movies,

    SUM(
        CASE
            WHEN budget = 0 THEN 1
            ELSE 0
        END
    ) AS Zero_Budget_Movies

FROM movies;
"""
pd.read_sql(query, engine)

,Total_Movies,Profitable_Movies,Loss_Making_Movies,Zero_Budget_Movies
0,4803,2585,1326,1037


In [23]:
# Querying the top 20 movies by revenue with their rank
query = """
SELECT
    movie_id,
    title,
    revenue,

    ROW_NUMBER() OVER (
        ORDER BY revenue DESC
    ) AS Revenue_Rank

FROM movies;
"""

pd.read_sql(query, engine).head(20)

,movie_id,title,revenue,Revenue_Rank
0,19995,Avatar,2.787965e+09,1
1,597,Titanic,1.845034e+09,2
2,24428,The Avengers,1.519558e+09,3
3,135397,Jurassic World,1.513529e+09,4
4,168259,Furious 7,1.506249e+09,5
5,99861,Avengers: Age of Ultron,1.405404e+09,6
6,109445,Frozen,1.274219e+09,7
7,68721,Iron Man 3,1.215440e+09,8
8,211672,Minions,1.156731e+09,9
9,271110,Captain America: Civil War,1.153304e+09,10


In [24]:
# Querying the top 20 movies by revenue with their rank using different ranking functions
query = """
SELECT
    movie_id,
    title,
    revenue,

    ROW_NUMBER() OVER (
        ORDER BY revenue DESC
    ) AS Row_Number,

    RANK() OVER (
        ORDER BY revenue DESC
    ) AS Revenue_Rank,

    DENSE_RANK() OVER (
        ORDER BY revenue DESC
    ) AS Dense_Revenue_Rank

FROM movies;
"""

pd.read_sql(query, engine).head(20)

,movie_id,title,revenue,Row_Number,Revenue_Rank,Dense_Revenue_Rank
0,19995,Avatar,2.787965e+09,1,1,1
1,597,Titanic,1.845034e+09,2,2,2
2,24428,The Avengers,1.519558e+09,3,3,3
3,135397,Jurassic World,1.513529e+09,4,4,4
4,168259,Furious 7,1.506249e+09,5,5,5
5,99861,Avengers: Age of Ultron,1.405404e+09,6,6,6
6,109445,Frozen,1.274219e+09,7,7,7
7,68721,Iron Man 3,1.215440e+09,8,8,8
8,211672,Minions,1.156731e+09,9,9,9
9,271110,Captain America: Civil War,1.153304e+09,10,10,10


## Ranking Functions

### ROW_NUMBER()

Assigns a unique sequential number to every row.

### RANK()

Rows with equal values receive the same rank, with gaps after ties.

### DENSE_RANK()

Rows with equal values receive the same rank, without gaps.

The appropriate function depends on the business question.

In [25]:
# Querying the top 20 movies by revenue with their rank within each year
query = """
SELECT
    movie_id,
    title,
    release_date,
    revenue,

    RANK() OVER (
        PARTITION BY strftime('%Y', release_date)
        ORDER BY revenue DESC
    ) AS Year_Revenue_Rank

FROM movies;
"""

pd.read_sql(query, engine).head(20)

,movie_id,title,release_date,revenue,Year_Revenue_Rank
0,380097,America Is Still the Place,NaN,0.0,1
1,3059,Intolerance,1916-09-04 00:00:00.000000,8394751.0,1
2,3060,The Big Parade,1925-11-05 00:00:00.000000,22000000.0,1
3,19,Metropolis,1927-01-10 00:00:00.000000,650422.0,1
4,65203,The Broadway Melody,1929-02-08 00:00:00.000000,4358000.0,1
5,905,Pandora's Box,1929-01-30 00:00:00.000000,0.0,2
6,22301,Hell's Angels,1930-11-15 00:00:00.000000,8000000.0,1
7,22649,A Farewell to Arms,1932-12-08 00:00:00.000000,25.0,1
8,3062,42nd Street,1933-02-02 00:00:00.000000,2281000.0,1
9,43595,She Done Him Wrong,1933-02-09 00:00:00.000000,2200000.0,2


In [26]:
# Querying the top 3 movies by revenue for each year
query = """
WITH ranked_movies AS (

    SELECT
        movie_id,
        title,
        release_date,
        revenue,

        RANK() OVER (
            PARTITION BY strftime('%Y', release_date)
            ORDER BY revenue DESC
        ) AS revenue_rank

    FROM movies

)

SELECT *
FROM ranked_movies
WHERE revenue_rank <= 3
ORDER BY release_date, revenue_rank;
"""

top_movies_per_year = pd.read_sql(query, engine)

top_movies_per_year.head(20)

,movie_id,title,release_date,revenue,revenue_rank
0,380097,America Is Still the Place,NaN,0.0,1
1,3059,Intolerance,1916-09-04 00:00:00.000000,8394751.0,1
2,3060,The Big Parade,1925-11-05 00:00:00.000000,22000000.0,1
3,19,Metropolis,1927-01-10 00:00:00.000000,650422.0,1
4,905,Pandora's Box,1929-01-30 00:00:00.000000,0.0,2
5,65203,The Broadway Melody,1929-02-08 00:00:00.000000,4358000.0,1
6,22301,Hell's Angels,1930-11-15 00:00:00.000000,8000000.0,1
7,22649,A Farewell to Arms,1932-12-08 00:00:00.000000,25.0,1
8,3062,42nd Street,1933-02-02 00:00:00.000000,2281000.0,1
9,43595,She Done Him Wrong,1933-02-09 00:00:00.000000,2200000.0,2


In [27]:
# Querying the yearly revenue and the revenue of the previous year
query = """
WITH yearly_revenue AS (

    SELECT
        strftime('%Y', release_date) AS release_year,
        SUM(revenue) AS total_revenue

    FROM movies

    GROUP BY strftime('%Y', release_date)

)

SELECT
    release_year,
    total_revenue,

    LAG(total_revenue) OVER (
        ORDER BY release_year
    ) AS previous_year_revenue

FROM yearly_revenue
ORDER BY release_year;
"""

yearly_revenue = pd.read_sql(query, engine)

yearly_revenue

,release_year,total_revenue,previous_year_revenue
0,NaN,0.000000e+00,NaN
1,1916,8.394751e+06,0.000000e+00
2,1925,2.200000e+07,8.394751e+06
3,1927,6.504220e+05,2.200000e+07
4,1929,4.358000e+06,6.504220e+05
...,...,...,...
86,2013,2.341149e+10,2.414171e+10
87,2014,2.412049e+10,2.341149e+10
88,2015,2.277502e+10,2.412049e+10
89,2016,1.446116e+10,2.277502e+10


In [28]:
# Querying the yearly revenue and the revenue of the previous year
query = """
WITH yearly_revenue AS (

    SELECT
        strftime('%Y', release_date) AS release_year,
        SUM(revenue) AS total_revenue

    FROM movies

    GROUP BY strftime('%Y', release_date)

),

revenue_comparison AS (

    SELECT
        release_year,
        total_revenue,

        LAG(total_revenue) OVER (
            ORDER BY release_year
        ) AS previous_year_revenue

    FROM yearly_revenue

)

SELECT
    release_year,
    total_revenue,
    previous_year_revenue,

    CASE
        WHEN previous_year_revenue IS NOT NULL
             AND previous_year_revenue <> 0

        THEN
            (
                (total_revenue - previous_year_revenue)
                * 100.0
                / previous_year_revenue
            )

        ELSE NULL
    END AS yoy_growth_percentage

FROM revenue_comparison

ORDER BY release_year;
"""

pd.read_sql(query, engine)

,release_year,total_revenue,previous_year_revenue,yoy_growth_percentage
0,NaN,0.000000e+00,NaN,NaN
1,1916,8.394751e+06,0.000000e+00,NaN
2,1925,2.200000e+07,8.394751e+06,162.068524
3,1927,6.504220e+05,2.200000e+07,-97.043536
4,1929,4.358000e+06,6.504220e+05,570.026537
...,...,...,...,...
86,2013,2.341149e+10,2.414171e+10,-3.024711
87,2014,2.412049e+10,2.341149e+10,3.028416
88,2015,2.277502e+10,2.412049e+10,-5.578105
89,2016,1.446116e+10,2.277502e+10,-36.504318


In [29]:
# Querying the yearly movie count and the movie count of the next year
query = """
WITH yearly_movies AS (

    SELECT
        strftime('%Y', release_date) AS release_year,
        COUNT(*) AS movie_count

    FROM movies

    GROUP BY strftime('%Y', release_date)

)

SELECT
    release_year,
    movie_count,

    LEAD(movie_count) OVER (
        ORDER BY release_year
    ) AS next_year_movie_count

FROM yearly_movies

ORDER BY release_year;
"""

pd.read_sql(query, engine)

,release_year,movie_count,next_year_movie_count
0,NaN,1,1.0
1,1916,1,1.0
2,1925,1,1.0
3,1927,1,2.0
4,1929,2,1.0
...,...,...,...
86,2013,231,238.0
87,2014,238,216.0
88,2015,216,104.0
89,2016,104,1.0


In [30]:
# Querying the cumulative revenue over the years
query = """
WITH yearly_revenue AS (

    SELECT
        strftime('%Y', release_date) AS release_year,
        SUM(revenue) AS yearly_revenue

    FROM movies

    GROUP BY strftime('%Y', release_date)

)

SELECT
    release_year,
    yearly_revenue,

    SUM(yearly_revenue) OVER (
        ORDER BY release_year
        ROWS BETWEEN UNBOUNDED PRECEDING
        AND CURRENT ROW
    ) AS cumulative_revenue

FROM yearly_revenue

ORDER BY release_year;
"""

pd.read_sql(query, engine)

,release_year,yearly_revenue,cumulative_revenue
0,NaN,0.000000e+00,0.000000e+00
1,1916,8.394751e+06,8.394751e+06
2,1925,2.200000e+07,3.039475e+07
3,1927,6.504220e+05,3.104517e+07
4,1929,4.358000e+06,3.540317e+07
...,...,...,...
86,2013,2.341149e+10,3.337412e+11
87,2014,2.412049e+10,3.578617e+11
88,2015,2.277502e+10,3.806367e+11
89,2016,1.446116e+10,3.950978e+11


In [31]:
# Querying the movies that have won an Oscar
query = """
SELECT
    m.movie_id,
    m.title

FROM movies m

WHERE EXISTS (

    SELECT 1

    FROM oscars o

    WHERE o.film = m.title

);
"""

movies_with_oscars = pd.read_sql(query, engine)

movies_with_oscars.head()

,movie_id,title
0,11,Star Wars
1,13,Forrest Gump
2,14,American Beauty
3,22,Pirates of the Caribbean: The Curse of the Bla...
4,28,Apocalypse Now


In [33]:
# Querying the movies that have not won an Oscar
query = """
SELECT
    m.movie_id,
    m.title

FROM movies m

WHERE NOT EXISTS (

    SELECT 1

    FROM oscars o

    WHERE o.film = m.title

);
"""

movies_without_oscars = pd.read_sql(
    query,
    engine
)

movies_without_oscars.head()

,movie_id,title
0,5,Four Rooms
1,12,Finding Nemo
2,16,Dancer in the Dark
3,18,The Fifth Element
4,19,Metropolis


In [34]:
# Querying unique actor IDs from the casts table
query = """
SELECT actor_id AS entity_id
FROM casts

UNION

SELECT actor_id AS entity_id
FROM casts;
"""

pd.read_sql(query, engine)

,entity_id
0,1
1,2
2,3
3,4
4,5
...,...
54583,1893203
54584,1893204
54585,1893205
54586,1893206


In [36]:
# Querying all actor IDs from the casts table, including duplicates
query = """
SELECT actor_id AS entity_id
FROM casts

UNION ALL

SELECT actor_id AS entity_id
FROM casts;
"""

pd.read_sql(query, engine)

,entity_id
0,1
1,1
2,1
3,1
4,2
...,...
212509,1893203
212510,1893204
212511,1893205
212512,1893206


## UNION vs UNION ALL

UNION removes duplicate rows.

UNION ALL preserves duplicate rows.

UNION ALL is generally preferable when duplicate removal is not required because it avoids the additional work required to deduplicate the result.

In [35]:
# Querying movie performance metrics
query = """
WITH movie_financials AS (

    SELECT
        movie_id,
        title,
        budget,
        revenue,
        revenue - budget AS profit

    FROM movies

),

movie_cast AS (

    SELECT
        movie_id,
        COUNT(DISTINCT actor_id) AS actor_count

    FROM casts

    GROUP BY movie_id

)

SELECT
    mf.movie_id,
    mf.title,
    mf.budget,
    mf.revenue,
    mf.profit,
    mc.actor_count

FROM movie_financials mf

LEFT JOIN movie_cast mc
    ON mf.movie_id = mc.movie_id

ORDER BY mf.revenue DESC;
"""

movie_performance = pd.read_sql(
    query,
    engine
)

movie_performance.head()

,movie_id,title,budget,revenue,profit,actor_count
0,19995,Avatar,237000000,2.787965e+09,2.550965e+09,83.0
1,597,Titanic,200000000,1.845034e+09,1.645034e+09,135.0
2,24428,The Avengers,220000000,1.519558e+09,1.299558e+09,115.0
3,135397,Jurassic World,150000000,1.513529e+09,1.363529e+09,28.0
4,168259,Furious 7,190000000,1.506249e+09,1.316249e+09,52.0


In [37]:
# Querying movie performance metrics by decade
query = """
SELECT
    (CAST(strftime('%Y', release_date) AS INTEGER) / 10) * 10
        AS release_decade,

    COUNT(*) AS movie_count,

    SUM(revenue) AS total_revenue,

    AVG(popularity) AS average_popularity

FROM movies

WHERE release_date IS NOT NULL

GROUP BY release_decade

ORDER BY release_decade;
"""

pd.read_sql(query, engine)

,release_decade,movie_count,total_revenue,average_popularity
0,1910,1,8.394751e+06,3.232447
1,1920,4,2.700842e+07,8.982580
2,1930,15,6.713090e+08,18.401319
3,1940,25,5.927508e+08,12.477554
4,1950,27,3.839422e+08,12.712991
5,1960,71,2.228629e+09,18.467904
6,1970,109,6.543658e+09,21.095282
7,1980,278,1.696337e+10,18.421812
8,1990,778,6.130123e+10,17.995693
9,2000,2048,1.566022e+11,18.159402


In [38]:
# Querying movie financials with ROI and profitability status
query = """
SELECT
    movie_id,
    title,
    budget,
    revenue,

    revenue - budget AS profit,

    CASE
        WHEN budget > 0
        THEN
            ((revenue - budget) * 100.0 / budget)
        ELSE NULL
    END AS roi_percentage,

    CASE
        WHEN revenue > budget THEN 'Profitable'
        WHEN revenue = budget THEN 'Break-even'
        WHEN revenue < budget THEN 'Loss'
        ELSE 'Unknown'
    END AS profitability_status

FROM movies;
"""

movie_financials = pd.read_sql(
    query,
    engine
)

movie_financials.head()

,movie_id,title,budget,revenue,profit,roi_percentage,profitability_status
0,5,Four Rooms,4000000,4300000.0,300000.0,7.500000,Profitable
1,11,Star Wars,11000000,775398007.0,764398007.0,6949.072791,Profitable
2,12,Finding Nemo,94000000,940335536.0,846335536.0,900.356953,Profitable
3,13,Forrest Gump,55000000,677945399.0,622945399.0,1132.627998,Profitable
4,14,American Beauty,15000000,356296601.0,341296601.0,2275.310673,Profitable


In [39]:
# Querying the top 10 movies by revenue
query = """
SELECT
    movie_id,
    title,
    revenue

FROM movies

ORDER BY revenue DESC

LIMIT 10;
"""

pd.read_sql(query, engine)

,movie_id,title,revenue
0,19995,Avatar,2.787965e+09
1,597,Titanic,1.845034e+09
2,24428,The Avengers,1.519558e+09
3,135397,Jurassic World,1.513529e+09
4,168259,Furious 7,1.506249e+09
5,99861,Avengers: Age of Ultron,1.405404e+09
6,109445,Frozen,1.274219e+09
7,68721,Iron Man 3,1.215440e+09
8,211672,Minions,1.156731e+09
9,271110,Captain America: Civil War,1.153304e+09


In [40]:
# Querying the top 10 movies by revenue with their rank
query = """
WITH ranked_movies AS (

    SELECT
        movie_id,
        title,
        revenue,

        ROW_NUMBER() OVER (
            ORDER BY revenue DESC
        ) AS revenue_rank

    FROM movies

)

SELECT *
FROM ranked_movies

WHERE revenue_rank <= 10;
"""

pd.read_sql(query, engine)

,movie_id,title,revenue,revenue_rank
0,19995,Avatar,2.787965e+09,1
1,597,Titanic,1.845034e+09,2
2,24428,The Avengers,1.519558e+09,3
3,135397,Jurassic World,1.513529e+09,4
4,168259,Furious 7,1.506249e+09,5
5,99861,Avengers: Age of Ultron,1.405404e+09,6
6,109445,Frozen,1.274219e+09,7
7,68721,Iron Man 3,1.215440e+09,8
8,211672,Minions,1.156731e+09,9
9,271110,Captain America: Civil War,1.153304e+09,10


In [41]:
# Querying the execution plan for a query that joins movies and casts
query = """
EXPLAIN QUERY PLAN

SELECT
    m.movie_id,
    m.title,
    c.actor_id

FROM movies m

LEFT JOIN casts c
    ON m.movie_id = c.movie_id;
"""

pd.read_sql(query, engine)

,id,parent,notused,detail
0,3,0,216,SCAN m
1,5,0,56,SEARCH c USING COVERING INDEX sqlite_autoindex...


## Query Optimization

Query optimization focuses on reducing unnecessary:

- Table scans
- Joins
- Sorting
- Aggregations
- Data movement
- Repeated calculations

Before optimizing a query, establish that the query is logically correct.

Optimization should not change the intended result.

In [42]:
# Function to validate the query result
def validate_query_result(df, key_column=None):

    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Missing values:")
    print(df.isna().sum().sum())

    if key_column:
        print(
            "Duplicate keys:",
            df[key_column].duplicated().sum()
        )

In [43]:
# Validate the query result
validate_query_result(
    movie_performance,
    key_column="movie_id"
)

Rows: 4803
Columns: 6
Missing values:
43
Duplicate keys: 0


# Advanced SQL Checklist

The following SQL concepts were implemented in this notebook:

- [x] Common Table Expressions
- [x] Multi-level CTEs
- [x] Subqueries
- [x] CASE expressions
- [x] Conditional aggregation
- [x] ROW_NUMBER()
- [x] RANK()
- [x] DENSE_RANK()
- [x] PARTITION BY
- [x] LAG()
- [x] LEAD()
- [x] Running totals
- [x] Top-N analysis
- [x] EXISTS
- [x] NOT EXISTS
- [x] UNION
- [x] UNION ALL
- [x] Advanced JOIN patterns
- [x] Date analysis
- [x] Financial calculations
- [x] Query validation
- [x] Query plan inspection
- [x] Query optimization concepts

## Key Engineering Principles

1. Understand table grain before writing a query.
2. Aggregate many-to-many relationships before joining where necessary.
3. Use CTEs to break complex transformations into logical stages.
4. Use window functions for ranking and period-over-period analysis.
5. Validate row counts and keys after complex transformations.
6. Avoid unnecessary SELECT * in production queries.
7. Filter data as early as logically appropriate.
8. Avoid unnecessary joins.
9. Use UNION ALL when deduplication is not required.
10. Optimize only after establishing query correctness.